# ParametricWakefield: Short-Range Wake of a Chain of RF Cavities

`ParametricWakefield` is the analytical fit of Zagorodnov, Tomin, Chen and Brinker (2021) to the longitudinal point-charge wake of a whole RF module, cavities together with their bellows and transitions. With $s = -z \ge 0$ the distance behind the source particle, Eq. 18 of the paper is

$$w(s) = \theta(s)\left(A\,e^{-\sqrt{k_0 s}} + B\,\frac{\cos\big((k_1 s)^{\alpha}\big)}{\sqrt{s} + k_2 s} + C\cos(k_3 s) + D\,\delta(s)\right).$$

| Term | Physical origin |
|---|---|
| $A\,e^{-\sqrt{k_0 s}}$ | Short-range wake of an infinite chain of cavities (Bane 2003) |
| $B\cos((k_1 s)^\alpha)/(\sqrt{s} + k_2 s)$ | Short-range wake of an isolated cavity, the diffraction model (Bane and Sands 1987), with a cosine for the oscillation seen with short bunches |
| $C\cos(k_3 s)$ | Long-range behaviour at the tail of a long bunch |
| $D\,\delta(s)$ | Step-out or collimator-like features |

The fit was made to wake potentials of Gaussian bunches with rms length from 10 µm to 2 mm, and Table IV of the paper gives coefficients for three structures of the European XFEL injector: the TESLA cryomodule A1, the 3.9 GHz third-harmonic module AH1, and the transverse deflecting structure TDS.

## Why this wake needs special treatment

The $B$ term diverges as $s^{-1/2}$ at the source particle, and the $\delta$ term has no point value at all. The wake therefore cannot be sampled at points, as a `TabularWakefield` of a smooth wake is. Instead the model provides its integrated wake through `integrated_wake(z)`

$$F(s) = \int_0^s w(s')\,ds',$$

and a table holds bin averages of the wake, the integrated Green function of Qiang et al. (2006),

$$K_0 = \frac{F(\Delta/2)}{\Delta}, \qquad K_k = \frac{F\big((k+\tfrac12)\Delta\big) - F\big((k-\tfrac12)\Delta\big)}{\Delta}.$$

$K_k$ is the wake at the centre of a bin from a unit charge spread uniformly over the bin $k$ bins ahead, and $K_0$ is the self term from the half of its own bin that lies ahead. Bmad, IMPACT-T, IMPACT-Z and `convolve_density` all halve the first row of a wake table for the self-kick, so the table stores $2K_0$ in its first row. With the $\delta$ term inside the first bin, it contributes $D\,\lambda(s)$ to the wake potential, the paper's resistive term with $R = D/c$.

`BinnedWakefield.from_wakefield` builds the fixed-width kernel from this integral. The model also uses bin averages automatically when convolving a density or tracking particles.

---
## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from beamphysics import ParticleGroup
from beamphysics.wakefields import (
    BinnedWakefield,
    ParametricWakefield,
    TabularWakefield,
    ZAGORODNOV_2021_TABLE_IV,
)

---
## The published coefficients

Table IV of the paper labels $A$ to $D$ in V/pC and $k_0$ to $k_3$ in 1/mm. `ZAGORODNOV_2021_TABLE_IV` holds the three sets as keyword arguments to `ParametricWakefield` in SI units, with the printed digits kept visible: $A = 711$ V/pC is stored as `711e12` V/C, and $k_0 = 1.94$ /mm as `1.94e3` /m.

In [ ]:
ZAGORODNOV_2021_TABLE_IV

Read literally, the labels of Table IV are not dimensionally consistent with Eq. 18. For $s$ in mm the coefficients would have to carry the units

$$[B] = \mathrm{V\,mm^{1/2}/pC}, \qquad [k_2] = \mathrm{mm^{-1/2}}, \qquad [D] = \mathrm{V\,mm/pC}.$$

The comparison with the paper's Fig. 7 below shows that the numbers reproduce the paper when the labels are converted literally, V/pC to V/C and 1/mm to 1/m, and the formula is evaluated in SI units with $s$ in metres. That is the conversion applied in `ZAGORODNOV_2021_TABLE_IV`. The wake of the whole module is spread over the structure `length`, which converts it to V/C/m, the unit the rest of this package uses. The TDS is 0.7 m long (Table 1 of the paper); the cryomodule lengths used here are nominal.

In [ ]:
lengths = {"A1": 12.2, "AH1": 10.5, "TDS": 0.7}

wakes = {
    name: ParametricWakefield(length=L, **ZAGORODNOV_2021_TABLE_IV[name])
    for name, L in lengths.items()
}
wakes["AH1"]

Coefficients quoted with $s$ in mm and $w$ in V/pC, as in much of the literature on cavity wakes, can be given with `from_mm_pC`; the SI constructor takes the whole-module coefficients directly. The AH1 set in mm units, for instance, has

$$B = 47.4\ \mathrm{V\,mm^{1/2}/pC}, \qquad k_2 = 6.36\ \mathrm{mm^{-1/2}}, \qquad D = 11.4\ \mathrm{V\,mm/pC}.$$

In [ ]:
ah1 = wakes["AH1"]
mm_per_m = 1e3
pC_per_C = 1e12
same = ParametricWakefield.from_mm_pC(
    length=ah1.length,
    A=ah1.A / pC_per_C,
    k0=ah1.k0 / mm_per_m,
    B=ah1.B * np.sqrt(mm_per_m) / pC_per_C,
    k1=ah1.k1 / mm_per_m,
    alpha=ah1.alpha,
    k2=ah1.k2 / np.sqrt(mm_per_m),
    C=ah1.C / pC_per_C,
    k3=ah1.k3 / mm_per_m,
    D=ah1.D * mm_per_m / pC_per_C,
)
same

---
## The point-charge wake

`wake(z)` returns the $A$, $B$ and $C$ terms per metre of structure; the $\delta$ term is carried by `integrated_wake`. At $z = 0$ the wake is infinite whenever $B \ne 0$. Multiplying by the length recovers the whole-module wake of the paper, in V/pC.

In [ ]:
s = np.geomspace(1e-6, 40e-3, 2000)

fig, ax = plt.subplots(figsize=(8, 4))
for name, wake in wakes.items():
    ax.plot(s * 1e3, wake.wake(-s) * wake.length * 1e-12, label=name)
ax.set_xscale("log")
ax.set_xlabel("Distance behind the source $s$ (mm)")
ax.set_ylabel("$w(s)$ (V/pC)")
ax.set_title("Point-charge wake of the whole module")
ax.axhline(0, color="k", lw=0.5)
ax.legend();

In [ ]:
wakes["AH1"].wake(0.0), wakes["AH1"].integrated_wake(-1e-6)

---
## Wake potentials of Gaussian bunches

`wake_potential(z, sigma)` evaluates the convolution of the wake with a Gaussian bunch of rms length `sigma` by quadrature that removes the singularity, in V/C/m per unit bunch charge. This is how the paper fits and checks the model, and it is the reference for every tabulated or binned result below.

### Comparison with the paper

Fig. 7 of the paper shows the wake potentials of the AH1 module from the electromagnetic solver ECHO for nine bunch lengths from 10 µm to 2 mm. The ECHO curves were digitized from the published figure, keeping only the points where a solid blue ECHO curve is clear of the red dashed fit, so the reference is independent of the model; the pixel resolution is about 0.03 in $s/\sigma_z$ and 6 V/pC in $W$.

In [ ]:
sigma_um, s_over_sigma, W_echo = np.loadtxt(
    "../data/zagorodnov_2021_fig7_echo.csv", delimiter=",", unpack=True
)
np.unique(sigma_um)

In [ ]:
wake = wakes["AH1"]

fig, axes = plt.subplots(3, 3, figsize=(13, 9), constrained_layout=True, sharex=True)
for ax, sig_um in zip(axes.flat, np.unique(sigma_um)):
    sigma = sig_um * 1e-6
    rows = sigma_um == sig_um
    x = np.linspace(-3, 5, 400)
    V = wake.wake_potential(-x * sigma, sigma) * wake.length * 1e-12
    ax.plot(
        s_over_sigma[rows], W_echo[rows], "C0", lw=3, alpha=0.5, label="ECHO, Fig. 7"
    )
    ax.plot(x, V, "C3--", label="ParametricWakefield")
    ax.fill_between(x, np.exp(-0.5 * x**2) * V.max(), alpha=0.1, color="k")
    ax.set_title(f"$\\sigma_z$ = {sig_um:g} µm")
    ax.axhline(0, color="k", lw=0.5)
for ax in axes[-1]:
    ax.set_xlabel("$s/\\sigma_z$")
for ax in axes[:, 0]:
    ax.set_ylabel("$W(s)$ (V/pC)")
axes[0, 0].legend()
fig.suptitle(
    "Longitudinal wake potential of the AH1 module: ECHO (digitized) and the model"
);

The model follows the ECHO curves to within a few V/pC rms, close to the pixel resolution of the figure, including the oscillation of the 10 and 15 µm bunches that the cosine of the $B$ term was introduced for. The largest relative deviation is on the tail of the 2 mm bunch, where the wake is smallest and the fit is at the edge of its range.

In [ ]:
def rms_residual(sig_um):
    sigma = sig_um * 1e-6
    rows = sigma_um == sig_um
    V = wake.wake_potential(-s_over_sigma[rows] * sigma, sigma) * wake.length * 1e-12
    return np.sqrt(np.mean((V - W_echo[rows]) ** 2))


# Columns: bunch length (µm), rms residual (V/pC)
np.array([(s, rms_residual(s)) for s in np.unique(sigma_um)]).round(1)

### Loss factor and rms spread

`loss_factor` and `rms_energy_spread` are the first two moments of the wake potential over the bunch, per unit charge and per metre.

In [ ]:
sig = np.geomspace(10e-6, 2e-3, 25)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for name, wake in wakes.items():
    kappa = np.array([wake.loss_factor(s) for s in sig]) * wake.length * 1e-12
    spread = np.array([wake.rms_energy_spread(s) for s in sig]) * wake.length * 1e-12
    ax1.plot(sig * 1e6, kappa, label=name)
    ax2.plot(sig * 1e6, spread, label=name)
for ax, label in ((ax1, "Loss factor (V/pC)"), (ax2, "Rms spread (V/pC)")):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("$\\sigma_z$ (µm)")
    ax.set_ylabel(label)
    ax.legend()

---
## Bin-averaged tables

`BinnedWakefield.from_wakefield` integrates a wake into fixed-width bins. The kernel has no pointwise `wake` method: its `dz` fixes the spacing for convolution and particle tracking, and `rebin` changes that spacing conservatively. Here AH1 is tabulated at 5 µm spacing over 4 mm. The rows nearest the source are shown as z (m) and W (V/C/m).

In [ ]:
wake = wakes["AH1"]
dz = 5e-6
zmax = 4e-3
n = int(round(zmax / dz)) + 1

table = BinnedWakefield.from_wakefield(wake, zmax=zmax, n=n)
np.column_stack((table.z_data[-4:], table.W_data[-4:]))

The stored source row is $2K_0 = F(\Delta/2)/(\Delta/2)$, the average over the causal half bin. The convolution halves this row to obtain $K_0$. Away from the source the bin averages follow the point wake closely, and they are finite everywhere.

In [ ]:
s = -table.z_data[::-1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    s[1:] * 1e3,
    wake.wake(-s[1:]) * wake.length * 1e-12,
    "k",
    lw=3,
    alpha=0.3,
    label="point wake $w(s)$",
)
ax.step(
    s * 1e3,
    table.W_data[::-1] * wake.length * 1e-12,
    where="mid",
    label=f"bin averages, $\\Delta$ = {dz * 1e6:g} µm",
)
ax.set_xscale("log")
ax.set_xlabel("$s$ (mm)")
ax.set_ylabel("V/pC")
ax.set_title("AH1: bin-averaged table and point wake")
ax.legend();

### The table reproduces the wake potential

Convolving the table with a Gaussian bunch sampled at the table spacing must reproduce the exact `wake_potential`, and the error must fall as the spacing shrinks. The same result comes from `wake.convolve_density` directly, which uses the bin averages whenever the model implements `integrated_wake`.

In [ ]:
def gaussian(z, sigma):
    return np.exp(-0.5 * (z / sigma) ** 2) / (np.sqrt(2 * np.pi) * sigma)


def tabulated_potential(wake, sigma, bins_per_sigma, half_width=8):
    dz = sigma / bins_per_sigma
    half = int(np.ceil(half_width * sigma / dz))
    z = dz * np.arange(-half, half + 1)
    density = gaussian(z, sigma)  # unit charge
    table = BinnedWakefield.from_wakefield(wake, zmax=2 * half * dz, n=2 * half + 1)
    return z, table.convolve_density(density, dz)

In [ ]:
sigma = 50e-6
z_ref = np.linspace(-5 * sigma, 5 * sigma, 500)
V_ref = wake.wake_potential(z_ref, sigma)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(
    -z_ref / sigma,
    V_ref * wake.length * 1e-12,
    "k",
    lw=3,
    alpha=0.3,
    label="exact quadrature",
)
for bins in (2, 5, 20):
    z, V = tabulated_potential(wake, sigma, bins)
    ax1.plot(
        -z / sigma, V * wake.length * 1e-12, label=f"table, {bins} bins per $\\sigma_z$"
    )
    err = np.interp(z_ref, z, V) - V_ref
    ax2.plot(
        -z_ref / sigma, err / np.abs(V_ref).max(), label=f"{bins} bins per $\\sigma_z$"
    )
ax1.set_xlim(-5, 5)
ax1.set_xlabel("$s/\\sigma_z$")
ax1.set_ylabel("$W(s)$ (V/pC)")
ax1.set_title(f"AH1, $\\sigma_z$ = {sigma * 1e6:g} µm")
ax1.legend()
ax2.set_xlim(-5, 5)
ax2.set_xlabel("$s/\\sigma_z$")
ax2.set_ylabel("Error relative to max |W|")
ax2.set_title("Tabulated minus exact")
ax2.legend();

In [ ]:
bins = np.array([1, 2, 4, 8, 16, 32, 64])
fig, ax = plt.subplots(figsize=(6, 4))
for name, w in wakes.items():
    V_ref = w.wake_potential(z_ref, sigma)
    errs = []
    for b in bins:
        z, V = tabulated_potential(w, sigma, b)
        errs.append(np.abs(np.interp(z_ref, z, V) - V_ref).max() / np.abs(V_ref).max())
    ax.loglog(bins, errs, "o-", label=name)
ax.loglog(bins, 0.3 / bins**2, "k--", label="$\\propto \\Delta^2$")
ax.set_xlabel("Bins per $\\sigma_z$")
ax.set_ylabel("Max error / max |W|")
ax.set_title(
    f"Convergence of the tabulated convolution, $\\sigma_z$ = {sigma * 1e6:g} µm"
)
ax.legend();

---
## The bin width must match

A bin-averaged table is correct only at the bin width the consuming code bins the bunch at. Bmad bins at the table spacing, so any spacing is right for Bmad. IMPACT-T and IMPACT-Z bin at their own mesh width and read the table by linear interpolation. If the mesh is finer than the table, the first row understates the singular self-bin coefficient; if it is coarser, the coefficient is overstated. Interpolation does not preserve the kernel integral.

Here a table built at $\Delta$ = 5 µm is applied by a code binning at `dz_code`. An explicit linear `TabularWakefield` simulates the external code treating the exported bins as point values. `BinnedWakefield` itself rejects a mismatched spacing; `rebin` changes that spacing while preserving the integral.

In [ ]:
sigma = 100e-6
delta = 5e-6
table = BinnedWakefield.from_wakefield(
    wake, zmax=16 * sigma, n=int(round(16 * sigma / delta)) + 1
)
sampled = TabularWakefield(table.z_data, table.W_data, kind="linear")

z_ref = np.linspace(-5 * sigma, 5 * sigma, 500)
V_ref = wake.wake_potential(z_ref, sigma)

In [ ]:
def applied_by_code(table, dz_code):
    half = int(np.ceil(8 * sigma / dz_code))
    z = dz_code * np.arange(-half, half + 1)
    density = gaussian(z, sigma)
    return z, table.convolve_density(density, dz_code)

In [ ]:
ratios = np.array([0.25, 0.5, 1, 2, 3, 4, 8])
mismatched, rebinned = [], []
for r in ratios:
    dz_code = r * delta
    z, V = applied_by_code(sampled, dz_code)
    mismatched.append(
        np.abs(np.interp(z_ref, z, V) - V_ref).max() / np.abs(V_ref).max()
    )
    z, V = applied_by_code(table.rebin(dz_code), dz_code)
    rebinned.append(np.abs(np.interp(z_ref, z, V) - V_ref).max() / np.abs(V_ref).max())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(ratios, mismatched, "o-", label="table at $\\Delta$ read at $dz$")
ax.loglog(ratios, rebinned, "s-", label="table rebinned to $dz$")
ax.set_xticks(ratios, [f"{r:g}" for r in ratios])
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.set_xlabel("$dz / \\Delta$")
ax.set_ylabel("Max error / max |W|")
ax.set_title(
    f"AH1, $\\sigma_z$ = {sigma * 1e6:g} µm, table spacing $\\Delta$ = {delta * 1e6:g} µm"
)
ax.legend();

When the code bins coarser than the table, reading the table by interpolation roughly doubles the error of the coarse binning itself; the rebinned table recovers the accuracy that the coarse bin width allows. The rebin is exact when `dz` is an odd multiple of $\Delta$, where the bin edges coincide. When the code bins finer than the table, the unchanged first row understates the singular self-bin coefficient. Rebinning preserves the integral but cannot recover structure below the original bin width.

---
## Tables for tracking codes

`to_bmad`, `to_impact_z` and `to_impact_t` write the first row unchanged, so a bin-averaged table is used by those codes as intended. For Bmad the table spacing is free. For the IMPACT codes rebin to the longitudinal mesh width of the element first, and mind the row limits of 5000 (IMPACT-Z) and 1000 (IMPACT-T).

In [ ]:
print("\n".join(table.to_bmad().splitlines()[:5]), "\n...")

In [ ]:
impact_table = table.rebin(dz=50e-6)
rfdata = impact_table.to_impact_z()
rfdata[:3]

### Reading bin-averaged tables

IMPACT-Z stores s (m), Wz (V/C/m), Wx and Wy (V/C/m²), but does not record whether the wake values are point samples or bin averages. Use `BinnedWakefield.from_impact_z` to recover the fixed spacing and bin averages; `TabularWakefield.from_impact_z` would instead treat the same values as interpolated point samples. Both readers accept a path or an array.

In [ ]:
restored_z = BinnedWakefield.from_impact_z(rfdata)
np.testing.assert_allclose(restored_z.W_data, impact_table.W_data)
restored_z.dz

IMPACT-T has no position column. Its writer returns the wake data and the table length (m); the reader needs both to reconstruct the spacing.

In [ ]:
impact_t_data, table_length = impact_table.to_impact_t()
restored_t = BinnedWakefield.from_impact_t(impact_t_data, length=table_length)
np.testing.assert_allclose(restored_t.W_data, impact_table.W_data)
restored_t.dz

---
## Kicks on a particle distribution

`ParticleGroup.apply_wakefield` takes the binned path for a singular wake: the charge is deposited on a grid by cloud-in-cell weighting, convolved with the bin-averaged wake, and interpolated back to the particles. A point-to-point sum would be dominated by the closest pairs of an $s^{-1/2}$ wake. The momentum change follows the exact wake potential of the bunch.

In [ ]:
rng = np.random.default_rng(0)
sigma = 100e-6
n_particles = 100_000
Q = 250e-12
z = rng.normal(0, sigma, n_particles)

P = ParticleGroup(
    data=dict(
        x=np.zeros(n_particles),
        px=np.zeros(n_particles),
        y=np.zeros(n_particles),
        py=np.zeros(n_particles),
        z=z,
        pz=np.full(n_particles, 130e6),
        t=np.zeros(n_particles),
        status=np.ones(n_particles, dtype=int),
        weight=np.full(n_particles, Q / n_particles),
        species="electron",
    )
)

In [ ]:
P2 = P.apply_wakefield(wake, length=wake.length, dz=sigma / 20)
dpz = P2.pz - P.pz

### Tracking with a fixed-width kernel

The same bunch can be tracked with a `BinnedWakefield` built at 5 µm spacing. Omitting `dz` in `apply_wakefield` uses the kernel's own spacing. The table covers 16 sigma, beyond the full extent of this sampled bunch, and its kicks should agree with the analytic model's binned calculation above. To choose another spacing, first call `rebin`; an explicit mismatched `dz` is rejected.

In [ ]:
tracking_bins = BinnedWakefield.from_wakefield(wake, zmax=16 * sigma, n=321)
P_binned = P.apply_wakefield(tracking_bins, length=wake.length)
binned_dpz = P_binned.pz - P.pz
np.testing.assert_allclose(binned_dpz, dpz, rtol=1e-8, atol=1e-6)
dpz = binned_dpz
tracking_bins.dz

In [ ]:
z_ref = np.linspace(-4 * sigma, 4 * sigma, 300)
expected = -wake.length * Q * wake.wake_potential(z_ref, sigma)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(-z[::20] / sigma, dpz[::20] * 1e-3, ".", ms=2, alpha=0.4, label="particles")
ax.plot(-z_ref / sigma, expected * 1e-3, "C3", label="$-Q L\\, W(s)$")
ax.set_xlim(-4, 4)
ax.set_xlabel("$s/\\sigma_z$")
ax.set_ylabel("$\\Delta p_z$ (keV/c)")
ax.set_title(f"AH1, {Q * 1e12:g} pC, $\\sigma_z$ = {sigma * 1e6:g} µm")
ax.legend();

---
## References

- I. Zagorodnov, S. Tomin, Y. Chen, F. Brinker, "Experimental validation of collective effects modeling at injector section of x-ray free-electron laser", Nucl. Instrum. Methods A 995, 165111 (2021), [arXiv:2010.10204](https://arxiv.org/abs/2010.10204). Eq. 18, Eqs. 23–25 and Table IV.
- K. L. F. Bane and M. Sands, "Wakefields of very short bunches in an accelerating cavity", SLAC-PUB-4441 (1987).
- K. L. F. Bane, "Short-range dipole wakefields in accelerating structures for the NLC", SLAC-PUB-9663 (2003).
- J. Qiang, S. Lidia, R. D. Ryne, C. Limborg-Deprey, "Three-dimensional quasistatic model for high brightness beam dynamics simulation", Phys. Rev. ST Accel. Beams 9, 044204 (2006).
- O. Zagorodnova and T. Limberg, "Impedance budget database for European XFEL", Proc. PAC'09, TU5RFP060.